# Optimistic vs Pessimistic Locking

## Overview
Concurrent access to shared data requires a strategy to prevent lost updates. The two fundamental approaches are pessimistic and optimistic locking.

**The lost update problem (without any locking):**
```
T1 reads balance = $5000
T2 reads balance = $5000
T1 writes balance = $5000 - $500 = $4500  (commits)
T2 writes balance = $5000 - $200 = $4800  (commits — T1's update is LOST)
Final balance: $4800 instead of $4300
```

**Pessimistic locking** — lock first, then read:
```sql
BEGIN;
SELECT balance FROM accounts WHERE account_id = 'ACC001' FOR UPDATE;
-- row locked; other writers queue here
UPDATE accounts SET balance = balance - 500 WHERE account_id = 'ACC001';
COMMIT;  -- lock released
```

**Optimistic locking** — read freely, check version at write:
```sql
-- Read (no lock):
SELECT balance, version FROM accounts WHERE account_id = 'ACC001';
-- (application logic)
-- Write with version guard:
UPDATE accounts SET balance = balance - 500, version = version + 1
WHERE account_id = 'ACC001' AND version = 1;
-- 0 rows updated → conflict → retry
```

---

In [ ]:
import sqlite3

def get_conn():
    c = sqlite3.connect(":memory:")
    c.execute("PRAGMA journal_mode=WAL")
    c.row_factory = sqlite3.Row
    return c

conn = get_conn()
conn.executescript("""
CREATE TABLE accounts (
    account_id TEXT PRIMARY KEY,
    owner_name TEXT NOT NULL,
    balance    REAL NOT NULL CHECK(balance >= 0),
    version    INTEGER NOT NULL DEFAULT 1,
    updated_at TEXT NOT NULL DEFAULT (datetime('now'))
);
INSERT INTO accounts VALUES
    ('ACC001','Aroha Ngata', 5000.0, 1, datetime('now')),
    ('ACC002','Liam Chen',   2500.0, 1, datetime('now'));
""")
conn.commit()

print("=== Pessimistic locking: lock before reading ===")
pessimistic_code = """
-- Pessimistic: assume conflicts WILL happen; lock rows before reading
-- Use when: high contention; conflicts are frequent; cost of retry is high

-- Step 1: lock the row at read time
BEGIN;
SELECT account_id, balance, version
FROM   accounts
WHERE  account_id = 'ACC001'
FOR UPDATE;                          -- row is now locked

-- Step 2: application logic (using the locked, guaranteed-fresh value)
-- Step 3: write
UPDATE accounts
SET    balance    = balance - 500,
       version    = version + 1,
       updated_at = datetime('now')
WHERE  account_id = 'ACC001';
COMMIT;                              -- lock released

-- No other transaction can modify ACC001 between SELECT and UPDATE.
-- Trade-off: reduces concurrency; other writers queue behind this lock.
"""
print(pessimistic_code)

print("=== Optimistic locking: check for conflict at write time ===")
optimistic_code = """
-- Optimistic: assume conflicts are RARE; skip locking; detect at write time
-- Use when: low contention; reads >> writes; retry cost is acceptable

-- Step 1: read without locking (plain SELECT)
BEGIN;
SELECT account_id, balance, version
FROM   accounts
WHERE  account_id = 'ACC001';       -- no FOR UPDATE

-- Step 2: application logic

-- Step 3: write WITH version check
UPDATE accounts
SET    balance    = balance - 500,
       version    = version + 1,
       updated_at = datetime('now')
WHERE  account_id = 'ACC001'
AND    version    = 1;              -- ← the version we read in Step 1

-- If another transaction updated the row between our read and write,
-- version is now 2, not 1. The WHERE clause matches 0 rows.
-- rows_affected = 0 → conflict detected → application retries from Step 1.
COMMIT;
"""
print(optimistic_code)


---
## Optimistic locking in Python

In [ ]:

print("=== Optimistic locking implemented in Python + SQLite ===")

import sqlite3, time

def get_conn():
    c = sqlite3.connect(":memory:")
    c.execute("PRAGMA journal_mode=WAL")
    c.row_factory = sqlite3.Row
    return c

conn = get_conn()
conn.executescript("""
CREATE TABLE accounts (
    account_id TEXT PRIMARY KEY, owner_name TEXT NOT NULL,
    balance REAL NOT NULL CHECK(balance >= 0),
    version INTEGER NOT NULL DEFAULT 1
);
INSERT INTO accounts VALUES ('ACC001','Aroha Ngata',5000.0,1);
""")
conn.commit()

def optimistic_withdraw(conn, account_id, amount, max_retries=5):
    """Withdraw amount using optimistic locking with version check."""
    for attempt in range(1, max_retries + 1):
        # Step 1: read current state (no lock)
        row = conn.execute(
            "SELECT balance, version FROM accounts WHERE account_id = ?",
            (account_id,)
        ).fetchone()
        if row is None:
            raise ValueError(f"Account {account_id} not found")

        balance, version = row["balance"], row["version"]
        if balance < amount:
            raise ValueError(f"Insufficient funds: ${balance:,.2f} < ${amount:,.2f}")

        # Step 2: write with version guard
        cursor = conn.execute(
            """UPDATE accounts
               SET balance = ?, version = version + 1
               WHERE account_id = ? AND version = ?""",
            (balance - amount, account_id, version)
        )
        conn.commit()

        if cursor.rowcount == 1:
            print(f"  Attempt {attempt}: withdrawal of ${amount:,.2f} succeeded "
                  f"(version {version} → {version+1})")
            return True
        else:
            print(f"  Attempt {attempt}: version conflict — retrying")
            time.sleep(0.01)

    raise RuntimeError(f"Failed after {max_retries} retries")

print("Optimistic withdrawal demo:")
optimistic_withdraw(conn, "ACC001", 500.0)
row = conn.execute("SELECT balance, version FROM accounts WHERE account_id='ACC001'").fetchone()
print(f"  Final state: balance=${row['balance']:,.2f}, version={row['version']}")


---
## CAS variants and strategy comparison

In [ ]:

print("=== CAS (Compare-And-Swap) and timestamp variants ===")
cas_variants = """
-- Version counter (integer):
UPDATE accounts
SET balance = ?, version = version + 1
WHERE account_id = ? AND version = ?

-- Timestamp (updated_at):
UPDATE accounts
SET balance = ?, updated_at = datetime('now')
WHERE account_id = ? AND updated_at = ?
-- Caution: timestamp precision (1 second in SQLite) means two updates in
-- the same second appear identical. Use integer version for safety.

-- Hash/checksum:
UPDATE accounts
SET balance = ?, row_hash = md5(balance::text)
WHERE account_id = ? AND row_hash = ?
-- Less common; useful when you cannot add a version column

-- PostgreSQL system column xmin (transaction ID):
-- Every row has an implicit xmin column holding the transaction that wrote it.
SELECT balance, xmin FROM accounts WHERE account_id = 'ACC001';
UPDATE accounts SET balance = ?
WHERE account_id = 'ACC001' AND xmin = ?
-- Caution: xmin wraps around after ~2 billion transactions; not portable.
"""
print(cas_variants)

print("=== Comparison: pessimistic vs optimistic ===")
comparison = [
    ("Mechanism",       "Lock row at read; release at commit",      "Version check at write; retry on mismatch"),
    ("Contention",      "Serialises all writers (queue behind lock)","Fails and retries on conflict"),
    ("Best for",        "High write contention; can't afford retry", "Low contention; reads >> writes"),
    ("Deadlock risk",   "Yes (if lock order not consistent)",        "No (no locks held)"),
    ("Latency",         "Higher under load (lock wait)",             "Lower under low contention; spiky under high"),
    ("DB support",      "Native (FOR UPDATE)",                       "Application-layer (version column needed)"),
    ("Finance example", "ATM withdrawal from shared account",        "User updating their own profile/preferences"),
]
print(f"  {'Factor':<16s}  {'Pessimistic':<42s}  Optimistic")
print("  " + "-"*85)
for factor, pess, opt in comparison:
    print(f"  {factor:<16s}  {pess:<42s}  {opt}")


---
## Common Pitfalls

**1. Not checking rowcount after an optimistic update**
`conn.execute('UPDATE ... WHERE version = ?')` returns a cursor. If `cursor.rowcount == 0`, a conflict occurred and the update was silently ignored. Without checking `rowcount`, the application proceeds as if the write succeeded when it did not. Always check `rowcount == 1` after every optimistic update.

**2. Using timestamp as the version column without sub-second precision**
A timestamp version with second-level precision (`datetime('now')` in SQLite, `TIMESTAMP` in PostgreSQL) treats two writes within the same second as identical versions. The second writer's version check passes even though a conflict occurred. Use an integer version counter instead -- it increments on every write with no precision ambiguity.

**3. Infinite retry loops without backoff or max-retry limits**
An optimistic retry loop without a maximum retry count can spin indefinitely under sustained high contention, pegging CPU and generating load. Always cap retries (e.g. 5 attempts) with exponential backoff and surface a proper error to the caller when retries are exhausted.

**4. Choosing optimistic locking for genuinely high-contention resources**
A shared account (joint account, corporate account) written by dozens of concurrent processes will have a very high conflict rate under optimistic locking. Every writer retries repeatedly, wasting work. High-contention writes belong with pessimistic locking (`FOR UPDATE`) so only one writer proceeds at a time without wasted round-trips.

**5. Reading outside the transaction for optimistic locking**
The read in optimistic locking should be inside the same transaction as the version-checked write. Reading outside a transaction, doing application work, then opening a new transaction for the write introduces a gap where the version check's isolation guarantee breaks down under some isolation levels.

**6. Not propagating the version to the API layer**
For web applications, the version read in Step 1 must survive the round trip to the browser and back. Include `version` in API responses and require it in update requests. If the front-end submits an update without a version, the backend cannot perform the version check and must either reject the request or fall back to pessimistic locking.


---
*sql_methods_library - Samantha McGarrigle*